In [ ]:
# SEIRS-SEI Malaria Model for Manaus with manual delay handling

This notebook implements a custom SEIRS-SEI model for malaria transmission in Manaus, Brazil, using an ODE approach with manual delay handling

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## Load Data

In [ ]:
DATA_DIR = '../../data_files/data'

climate_data = pd.read_csv(DATA_DIR + '/climate_api_data_2016_2024.csv')
cases_data = pd.read_csv(DATA_DIR + '/sivep_notification_data/treated_malaria_notification_data/cumulative_manaus_cases_2016_2023.csv')

pop_data = pd.read_csv(DATA_DIR + '/ibge_manaus_population_data_2016_2024.csv')
pop_data.rename(columns={'index': 'date'}, inplace=True)

defor_data = pd.read_csv(DATA_DIR + '/deter_notification_data/treated_deter_deforestation_data_2016_2024.csv')
fires_data = pd.read_csv(DATA_DIR + '/inpe_fire_counts_data_2016_2024.csv')

climate_data['date'] = pd.to_datetime(climate_data['date'])
cases_data['date'] = pd.to_datetime(cases_data['date'])
pop_data['date'] = pd.to_datetime(pop_data['date'])
defor_data['date'] = pd.to_datetime(defor_data['date'])
fires_data['date'] = pd.to_datetime(fires_data['date'])

# Filter data to 2017-01-01 to 2023-12-31
start_date = pd.to_datetime('2017-01-01')
end_date = pd.to_datetime('2023-12-31')
climate_data = climate_data[(climate_data['date'] >= start_date) & (climate_data['date'] <= end_date)].reset_index(drop=True)
cases_data = cases_data[(cases_data['date'] >= start_date) & (cases_data['date'] <= end_date)].reset_index(drop=True)
pop_data = pop_data[(pop_data['date'] >= start_date) & (pop_data['date'] <= end_date)].reset_index(drop=True)
defor_data = defor_data[(defor_data['date'] >= start_date) & (defor_data['date'] <= end_date)].reset_index(drop=True)
fires_data = fires_data[(fires_data['date'] >= start_date) & (fires_data['date'] <= end_date)].reset_index(drop=True)

print("Data loaded and filtered successfully!")
print("Climate data:", len(climate_data), "days")
print("Cases data:", len(cases_data), "days")
print("Pop data:", len(pop_data), "days")
print("Deforestation data:", len(defor_data), "days")
print("Forest fires data:", len(fires_data), "days")